# تشغيل النموذج كـ API من Colab — Serve your model as an API

النوتبوك دي بتشغّل نموذجك المدرّب (`darklink56/qwen-coder-1.5b-lora-out`) على
كارت T4 المجاني وبتطلع لك **رابط عام (gradio.live)** — أي تطبيق عندك
(Expo / ويب / أي حاجة) يقدر يكلم الرابط ده كـ API وياخد ردود النموذج.

- التكلفة: **0** — Colab مجاني + رابط Gradio العام مجاني.
- الرابط شغال **طول ما الجلسة دي مفتوحة بس**؛ لو الجلسة قفلت شغّل النوتبوك
  تاني وهيطلع رابط جديد.
- السرعة: على T4 الردود سريعة بجد، أسرع بكتير من أي استضافة CPU مجانية.

> المتطلبات: Runtime على **T4 GPU**، وتسجيل دخول Hugging Face زي نوتبوك
> التدريب بالظبط (سر `HF_TOKEN` أو خانة التوكن التفاعلية).


In [ ]:
# 1) Check the GPU — لازم T4
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU found! Go to Runtime → Change runtime type → T4 GPU, then run again.")


In [ ]:
# 2) Install — نفس الإصدارات المثبتة بتاعة التدريب + gradio
!pip install -q "trl==0.29.1" "transformers==4.57.6" "peft==0.22.0" "accelerate==1.16.0" "bitsandbytes==0.50.2" "gradio==5.50.0"
print("✅ Installed")


In [ ]:
# 3) Log in to Hugging Face — نفس خلية التدريب بالظبط
import os

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

from huggingface_hub import login
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)
    print("✅ Logged in via Colab Secrets (HF_TOKEN)")
else:
    print("ℹ️ مفيش HF_TOKEN في الأسرار — هيظهر لك خانة توكن دلوقتي، الصق توكن الـ Write فيها:")
    login()
    print("✅ Logged in interactively")


In [ ]:
# 4) Load the trained model on the GPU (4-bit) + merge the adapter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER = "darklink56/qwen-coder-1.5b-lora-out"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
tokenizer.pad_token = "<|endoftext|>"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, ADAPTER)
model = model.merge_and_unload()
model.eval()
print("✅ Model ready on GPU")


In [ ]:
# 5) The API app — صفحة تجربة + API باسم generate
import torch
import gradio as gr


def generate(prompt: str, max_new_tokens: int = 200) -> str:
    if not prompt or not prompt.strip():
        return "اكتب سؤال أو مهمة كود الأول."
    messages = [{"role": "user", "content": prompt.strip()}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


demo = gr.Interface(
    fn=generate,
    inputs=[
        gr.Textbox(label="Prompt", lines=3, placeholder="Write a Python function that..."),
        gr.Slider(32, 1024, value=200, step=16, label="Max new tokens"),
    ],
    outputs=gr.Textbox(label="Answer", lines=10),
    title="darklink56 Code Model — API",
    api_name="generate",
)
print("✅ App built — شغّل الخلية اللي بعدها عشان يطلع الرابط العام")


In [ ]:
# 6) Launch with a public link — هيطبع لك رابط https://....gradio.live
# الرابط ده هو عنوان الـ API بتاعك طول ما الجلسة مفتوحة:
#   POST  {الرابط}/gradio_api/call/generate        {"data": ["سؤالك", 200]}
#   GET   {الرابط}/gradio_api/call/generate/{event_id}
# مثال كود الربط من Expo موجود في space/README.md في الريبو (نفس الفكرة،
# بدّل عنوان الـ Space بعنوان gradio.live اللي هيظهر تحت).
demo.launch(share=True)
